<a href="https://colab.research.google.com/github/SeiDra/lending-club-prediction/blob/scindage-du-notebook-FeatureEngineering%2FModeling/du_sda_ml2_P3_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PROJET 7 : Loan Default Prediction
Partie N°3 — Feature Engineering

Contenu :
- Import des données nettoyées (sortie du P2)
- Identification des types de variables
- Binary Encoding pour les variables binaires
- Train / Test Split (stratifié)
- Target Encoding pour les variables multi-catégorielles (fit sur train)
- Normalisation des variables numériques (fit sur train)
- Sauvegarde des jeux train/test et des transformateurs

In [ ]:
import joblib
import numpy as np
import pandas as pd
from category_encoders import TargetEncoder
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_parquet("DATA/cleaned_data_for_modeling.parquet")

# Lecture du nom de la cible depuis le fichier de configuration
with open("CONFIG/target_config.txt", "r") as f:
    target_col = f.read().strip()

print(f"Cible : {target_col}")
print(f"Dimensions du dataset : {df.shape}")
print(f"\nDistribution de la cible :")
print(df[target_col].value_counts())
print(df[target_col].value_counts(normalize=True) * 100)

In [ ]:
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
num_cols = [col for col in df.select_dtypes(exclude=['object']).columns.tolist()
            if col != target_col]

binary_cat_cols = [col for col in cat_cols if df[col].nunique() <= 2]
multi_cat_cols = [col for col in cat_cols if df[col].nunique() > 2]

print(f"Variables numériques : {len(num_cols)}")
print(f"Variables catégorielles binaires : {len(binary_cat_cols)} → {binary_cat_cols}")
print(f"Variables catégorielles multi : {len(multi_cat_cols)} → {multi_cat_cols}")

In [ ]:
# Le binary encoding (get_dummies) ne dépend pas de la cible,
# donc il peut être fait avant le split sans risque de leakage.

original_cols = df.columns.tolist()
df = pd.get_dummies(df, columns=binary_cat_cols, drop_first=True)

# Identifier les nouvelles colonnes créées par le binary encoding
new_binary_cols = [col for col in df.columns if col not in original_cols]
print(f"Nouvelles colonnes binaires créées : {new_binary_cols}")
print(f"Dimensions après binary encoding : {df.shape}")

df.head().T

In [ ]:
# Le split DOIT être fait AVANT le target encoding et la normalisation
# pour éviter le data leakage.

stratified = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_idx, test_idx in stratified.split(df, df[target_col]):
    train_df = df.iloc[train_idx].reset_index(drop=True)
    test_df = df.iloc[test_idx].reset_index(drop=True)

# Séparation X / y
train_y = train_df[[target_col]]
test_y = test_df[[target_col]]

train_X = train_df.drop(columns=[target_col])
test_X = test_df.drop(columns=[target_col])

print(f"Train : {train_X.shape} | Ratio défaut : {train_y[target_col].mean():.2%}")
print(f"Test  : {test_X.shape}  | Ratio défaut : {test_y[target_col].mean():.2%}")

In [ ]:
encoder = TargetEncoder(cols=multi_cat_cols, smoothing=0.2)

# Fit UNIQUEMENT sur le train
train_X[multi_cat_cols] = encoder.fit_transform(train_X[multi_cat_cols], train_y)

# Appliquer les mêmes règles au test
test_X[multi_cat_cols] = encoder.transform(test_X[multi_cat_cols])

print("Target Encoding appliqué avec succès.")
train_X.head().T

In [ ]:
scaler = StandardScaler()

# Fit UNIQUEMENT sur le train
train_X[num_cols] = scaler.fit_transform(train_X[num_cols])

# Appliquer les mêmes règles au test
test_X[num_cols] = scaler.transform(test_X[num_cols])

print("Normalisation appliquée avec succès.")
train_X.head().T

In [ ]:
print(" VÉRIFICATIONS ")
print(f"NaN dans train_X : {train_X.isnull().sum().sum()}")
print(f"NaN dans test_X  : {test_X.isnull().sum().sum()}")
print(f"Colonnes train_X : {train_X.shape[1]}")
print(f"Colonnes test_X  : {test_X.shape[1]}")
print(f"Colonnes identiques : {list(train_X.columns) == list(test_X.columns)}")

In [ ]:
train_X.to_parquet("DATA/03_train_X.parquet")
train_y.to_parquet("DATA/03_train_y.parquet")
test_X.to_parquet("DATA/03_test_X.parquet")
test_y.to_parquet("DATA/03_test_y.parquet")

# Sauvegarder les transformateurs pour une réutilisation future (production)
joblib.dump(encoder, "DATA/03_encoder.pkl")
joblib.dump(scaler, "DATA/03_scaler.pkl")

print(" Tous les fichiers ont été sauvegardés avec succès !")
print("   - DATA/03_train_X.parquet")
print("   - DATA/03_train_y.parquet")
print("   - DATA/03_test_X.parquet")
print("   - DATA/03_test_y.parquet")
print("   - DATA/03_encoder.pkl")
print("   - DATA/03_scaler.pkl")